In [ ]:
from sklearn.metrics import r2_score ,mean_absolute_error ,mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge , Lasso
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import pandas as pd 
import numpy as np  

In [2]:
df = pd.read_csv('data/stud.csv')
df

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75
...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95
996,male,group C,high school,free/reduced,none,62,55,55
997,female,group C,high school,free/reduced,completed,59,71,65
998,female,group D,some college,standard,completed,68,78,77


In [3]:
df['total_score']= df['math_score'] + df['reading_score'] + df['writing_score']
df['avg_score'] = df['total_score'] /3

In [4]:
df


,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,total_score,avg_score
0,female,group B,bachelor's degree,standard,none,72,72,74,218,72.666667
1,female,group C,some college,standard,completed,69,90,88,247,82.333333
2,female,group B,master's degree,standard,none,90,95,93,278,92.666667
3,male,group A,associate's degree,free/reduced,none,47,57,44,148,49.333333
4,male,group C,some college,standard,none,76,78,75,229,76.333333
...,...,...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95,282,94.000000
996,male,group C,high school,free/reduced,none,62,55,55,172,57.333333
997,female,group C,high school,free/reduced,completed,59,71,65,195,65.000000
998,female,group D,some college,standard,completed,68,78,77,223,74.333333


In [5]:
X = df.drop('avg_score' , axis=1)
X

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,total_score
0,female,group B,bachelor's degree,standard,none,72,72,74,218
1,female,group C,some college,standard,completed,69,90,88,247
2,female,group B,master's degree,standard,none,90,95,93,278
3,male,group A,associate's degree,free/reduced,none,47,57,44,148
4,male,group C,some college,standard,none,76,78,75,229
...,...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95,282
996,male,group C,high school,free/reduced,none,62,55,55,172
997,female,group C,high school,free/reduced,completed,59,71,65,195
998,female,group D,some college,standard,completed,68,78,77,223


In [6]:
y = df['avg_score']
y

0      72.666667
1      82.333333
2      92.666667
3      49.333333
4      76.333333
         ...    
995    94.000000
996    57.333333
997    65.000000
998    74.333333
999    83.000000
Name: avg_score, Length: 1000, dtype: float64

In [7]:
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder , LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_tranformer = OneHotEncoder()

preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder" , oh_tranformer , cat_features),
        ("StandardScaler" ,numeric_transformer , num_features)
    ]
)


In [8]:
X = preprocessor.fit_transform(X)   

In [9]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42) 

In [13]:
def evaluate_model(true , predicted):
    mae = mean_absolute_error(true , predicted)
    mse = mean_squared_error(true , predicted)
    rmse = np.sqrt(mean_squared_error(true , predicted))
    r2 = r2_score(true , predicted)
    return mae , rmse , r2

In [15]:
models ={
    "Linear Regression":LinearRegression(),
    "Ridge":Ridge() ,
    "Lasso":Lasso() ,
    "K_Neighbors_Regressor":KNeighborsRegressor(),
    "Decision_Tree_Regressor":DecisionTreeRegressor(),
    "Random_Forest_Regressor":RandomForestRegressor(),
    "XG_Regresseor":XGBRegressor(),
    "Cat_Boost_Regressor":CatBoostRegressor(),
    "ADA_Boost_Regressor":AdaBoostRegressor(),
}

model_list =[]
r2_list = []

for i in range (len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train , y_train )

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae , model_train_rmse ,model_train_r2 = evaluate_model(y_train ,y_train_pred)
    model_test_mae , model_test_rmse ,model_test_r2 = evaluate_model(y_test ,y_test_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model performance for training set')
    print("- root mean squared error: {:.4f}".format(model_train_rmse))
    print("- Mean absolute error :{:.4f}".format(model_train_mae))
    print("- R2 Score : {:.4f}" .format(model_train_r2))

    print("------------------------------------")

    
    print('Model performance for test set')
    print("- root mean squared error: {:.4f}".format(model_test_rmse))
    print("- Mean absolute error :{:.4f}".format(model_test_mae))
    print("- R2 Score : {:.4f}" .format(model_test_r2))
    r2_list.append(model_test_r2)

    print("\n")


Linear Regression
Model performance for training set
- root mean squared error: 0.0000
- Mean absolute error :0.0000
- R2 Score : 1.0000
------------------------------------
Model performance for test set
- root mean squared error: 0.0000
- Mean absolute error :0.0000
- R2 Score : 1.0000


Ridge
Model performance for training set
- root mean squared error: 0.0059
- Mean absolute error :0.0048
- R2 Score : 1.0000
------------------------------------
Model performance for test set
- root mean squared error: 0.0064
- Mean absolute error :0.0049
- R2 Score : 1.0000


Lasso
Model performance for training set
- root mean squared error: 1.0090
- Mean absolute error :0.8075
- R2 Score : 0.9949
------------------------------------
Model performance for test set
- root mean squared error: 1.0557
- Mean absolute error :0.8312
- R2 Score : 0.9948


K_Neighbors_Regressor
Model performance for training set
- root mean squared error: 1.9816
- Mean absolute error :1.5570
- R2 Score : 0.9803
----------

In [16]:
pd.DataFrame(list(zip(model_list ,r2_list)) ,columns=['Model Name' ,'R2_score']).sort_values(by=["R2_score"] , ascending=False)

,Model Name,R2_score
0,Linear Regression,1.000000
1,Ridge,1.000000
4,Decision_Tree_Regressor,0.997846
6,XG_Regresseor,0.997431
5,Random_Forest_Regressor,0.996539
7,Cat_Boost_Regressor,0.994961
2,Lasso,0.994801
8,ADA_Boost_Regressor,0.990064
3,K_Neighbors_Regressor,0.966842


In [17]:
lin_model = LinearRegression(fit_intercept=True )
lin_model = lin_model.fit(X_train , y_train )
y_pred = lin_model.predict(X_test)
score = r2_score(y_test , y_pred) *100

In [18]:
pred_df = pd.DataFrame(
    {
        'Actual value':y_test,
        'Predicted value':y_pred,
        'Difference':y_test-y_pred ,

    }
)

pred_df

,Actual value,Predicted value,Difference
521,87.000000,87.000000,0.000000e+00
737,64.000000,64.000000,7.105427e-15
740,75.000000,75.000000,1.421085e-14
660,74.666667,74.666667,2.842171e-14
411,81.666667,81.666667,2.842171e-14
...,...,...,...
408,55.000000,55.000000,0.000000e+00
332,57.000000,57.000000,1.421085e-14
208,77.000000,77.000000,0.000000e+00
613,72.000000,72.000000,1.421085e-14
